# 02 — Baselines: el piso a superar

**TP Final · Aprendizaje de Máquina I (CEIA-FIUBA)** · Jaime Pinzón (a2629)

## ¿Por qué un baseline, y por qué estos?

Un baseline responde la pregunta *"¿cuánto se puede lograr sin aprender casi nada?"*. Si un modelo complejo no supera con claridad este piso, su costo no se justifica — esa comparación es el eje del informe final.

- **`DummyRegressor(strategy="mean")`**: predice siempre la media del train. Es el predictor constante que minimiza el error cuadrático.
- **`DummyRegressor(strategy="median")`**: predice siempre la mediana. Es el predictor constante que minimiza el **MAE** — nuestra métrica principal — *respecto de la distribución sobre la que se calcula* (matiz que resultará importante, ver más abajo).
- **Regresión lineal (OLS)**: la referencia paramétrica más simple que sí usa las features. Marca cuánta señal lineal hay en los datos.

**Nota metodológica:** la cátedra pide explícitamente NO usar regresión logística como baseline. Además de la indicación, hay una razón técnica: la regresión logística es un **clasificador** (modela probabilidades de clases discretas); nuestro problema es de **regresión** sobre días de internación, donde el análogo correcto es la regresión lineal.

Los tres baselines usan el mismo preprocesador que usarán todos los modelos (`build_preprocessor`), sin escalado: el Dummy ignora las features y OLS es invariante a transformaciones afines de las columnas.

In [1]:
import time

import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline

from src.config import DIR_PROCESSED, SEED, TARGET
from src.evaluacion import evaluar, registrar
from src.pipelines import build_preprocessor

X_train = pd.read_parquet(DIR_PROCESSED / "X_train.parquet")
X_test = pd.read_parquet(DIR_PROCESSED / "X_test.parquet")
y_train = pd.read_parquet(DIR_PROCESSED / "y_train.parquet")[TARGET]
y_test = pd.read_parquet(DIR_PROCESSED / "y_test.parquet")[TARGET]

print(f"train: {X_train.shape} | test: {X_test.shape}")
print(f"media del train: {y_train.mean():.3f} dias | mediana: {y_train.median():.1f} dias")

train: (53756, 8) | test: (20354, 8)
media del train: 4.087 dias | mediana: 3.0 dias


In [2]:
modelos = {
    "baseline_media": DummyRegressor(strategy="mean"),
    "baseline_mediana": DummyRegressor(strategy="median"),
    "regresion_lineal": LinearRegression(),
}
notas = {
    "baseline_media": "DummyRegressor(mean) sobre el train filtrado; predictor constante",
    "baseline_mediana": "DummyRegressor(median) sobre el train filtrado",
    "regresion_lineal": "OLS sobre las 17 features del preprocesador comun",
}

# Mismo protocolo de validacion que usaran TODAS las familias: KFold-5, seed 42,
# solo sobre train. Es la columna con la que el NB07 elegira el ganador.
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

resultados = {}
for nombre, estimador in modelos.items():
    pipe = Pipeline([
        ("preprocesador", build_preprocessor(scale=False)),
        ("modelo", estimador),
    ])
    # n_jobs=1: son modelos triviales, el overhead de paralelizar supera la ganancia
    mae_cv = -cross_val_score(
        pipe, X_train, y_train, cv=cv,
        scoring="neg_mean_absolute_error", n_jobs=1,
    ).mean()

    t0 = time.perf_counter()
    pipe.fit(X_train, y_train)
    tiempo = time.perf_counter() - t0

    metricas = evaluar(pipe, X_test, y_test)
    metricas["mae_cv"] = float(mae_cv)
    metricas["tiempo_s"] = round(tiempo, 2)
    registrar(nombre, metricas, notas=notas[nombre])
    resultados[nombre] = metricas

pd.DataFrame(resultados).T[["mae_cv", "mae", "rmse", "r2", "tiempo_s", "pred_test_s"]].round(4)


,mae_cv,mae,rmse,r2,tiempo_s,pred_test_s
baseline_media,2.0689,2.2805,3.0009,-0.0107,0.05,0.01
baseline_mediana,2.0281,2.2929,3.2952,-0.2187,0.05,0.01
regresion_lineal,1.7081,1.8616,2.4666,0.3171,0.11,0.03


In [3]:
# El registro usa UPSERT por nombre: re-ejecutar este notebook NO duplica filas.
from src.evaluacion import RUTA_METRICAS

registrar("baseline_media", resultados["baseline_media"], notas=notas["baseline_media"])
tabla = pd.read_csv(RUTA_METRICAS)
assert tabla["modelo"].is_unique, "hay modelos duplicados en metricas.csv"
print(f"metricas.csv: {len(tabla)} filas, sin duplicados")
tabla.round(4)

metricas.csv: 10 filas, sin duplicados


,modelo,mae_cv,mae,rmse,r2,tiempo_s,pred_test_s,notas
0,baseline_media,2.0689,2.2805,3.0009,-0.0107,0.05,0.01,DummyRegressor(mean) sobre el train filtrado; ...
1,baseline_mediana,2.0281,2.2929,3.2952,-0.2187,0.05,0.01,DummyRegressor(median) sobre el train filtrado
2,regresion_lineal,1.7081,1.8616,2.4666,0.3171,0.11,0.03,OLS sobre las 17 features del preprocesador comun
3,knn,1.6775,1.8341,2.4731,0.3136,0.13,9.07,"k=80, weights=distance, p=1; grilla 32 configs..."
4,svr_lineal,1.6795,1.8343,2.5168,0.2891,0.28,0.02,"LinearSVR C=0.01, eps=0.5; train completo; gri..."
5,svr_rbf,1.6283,1.7687,2.4425,0.3304,133.98,101.61,"SVR-RBF ['C=8.598', 'epsilon=0.4397', 'gamma=0..."
6,arbol_podado,1.7362,1.8585,2.4829,0.3081,6.93,0.02,ccp_alpha=9.62e-04 por CV5 sobre 25 candidatos...
7,random_forest,1.6530,1.7980,2.3994,0.3539,9.98,0.37,"RF {'n_estimators': 500, 'max_features': 0.370..."
8,xgboost,1.6073,1.7477,2.4175,0.3441,13.15,0.13,"XGB objective=reg:absoluteerror, {'learning_ra..."
9,baseline_mediana_sin_filtrar,2.0429,2.2596,3.0111,-0.0176,0.00,0.00,Constante = mediana del train SIN filtro IQR (...


## ¿Por qué la media le ganó a la mediana en MAE, si "la mediana minimiza el MAE"?

La propiedad es cierta pero **relativa a la distribución sobre la que se evalúa**. Los baselines se ajustan sobre el train **filtrado por IQR** y se evalúan sobre el test **sin filtrar**. El filtro corrió la mediana del train a 3 días, mientras que la del test es 4: la constante 3 queda sistemáticamente corta frente a la cola alta que el test conserva.

La celda siguiente cuantifica el fenómeno y, sobre todo, **identifica su causa real**. La mediana del train **antes** del filtro IQR también es 4,0 — es decir, la constante óptima era accesible desde el principio usando únicamente información de entrenamiento. No hacía falta mirar el test para encontrarla: **lo que degradaba al baseline de mediana no era la asimetría de la distribución, era el filtro heredado auditado en el notebook 01**.

Esto obliga a corregir el piso de comparación de todo el trabajo. El baseline honesto más fuerte no es `DummyRegressor(mean)` con MAE 2,2805, sino **la mediana del train sin filtrar, con MAE 2,2596** — y contra ese piso, más exigente, se juzgan los modelos de los notebooks 03–06.

In [4]:
# Evidencia numerica del fenomeno media-vs-mediana, y de su verdadera causa
from src.preprocesamiento import cargar_y_limpiar, split_estratificado

print(f"y_train (FILTRADO): media={y_train.mean():.4f}  mediana={y_train.median():.1f}  "
      f"skew={y_train.skew():.3f}  max={y_train.max():.0f}")
print(f"y_test  (sin filtrar): media={y_test.mean():.4f}  mediana={y_test.median():.1f}  "
      f"skew={y_test.skew():.3f}  max={y_test.max():.0f}")
print()

# El train SIN filtrar: mismo split, sin el filtro IQR. Es informacion de
# entrenamiento legitima -- no mira el test en ningun momento.
X_todo, y_todo = cargar_y_limpiar()
_, _, y_train_sin_filtrar, _ = split_estratificado(X_todo, y_todo)
mediana_sin_filtrar = y_train_sin_filtrar.median()

print(f"y_train SIN filtrar: media={y_train_sin_filtrar.mean():.4f}  "
      f"mediana={mediana_sin_filtrar:.1f}  max={y_train_sin_filtrar.max():.0f}")
print()

mae_mediana_filtrada = (y_test - y_train.median()).abs().mean()
mae_media_filtrada = (y_test - y_train.mean()).abs().mean()
mae_mediana_sin_filtrar = (y_test - mediana_sin_filtrar).abs().mean()
mae_mediana_test = (y_test - y_test.median()).abs().mean()

print(f"MAE(test) prediciendo la mediana del train FILTRADO ({y_train.median():.1f}):      "
      f"{mae_mediana_filtrada:.4f}")
print(f"MAE(test) prediciendo la media del train filtrado ({y_train.mean():.3f}):    "
      f"{mae_media_filtrada:.4f}")
print(f"MAE(test) prediciendo la mediana del train SIN filtrar ({mediana_sin_filtrar:.1f}):   "
      f"{mae_mediana_sin_filtrar:.4f}   <- PISO HONESTO")
print(f"MAE(test) prediciendo la mediana del TEST ({y_test.median():.1f}):              "
      f"{mae_mediana_test:.4f}   <- optimo entre constantes")
print()
print("La mediana del train sin filtrar (4.0) coincide con la del test y alcanza el")
print("optimo entre constantes SIN mirar el test: el optimo era accesible todo el tiempo.")
print("Lo que degradaba al baseline de mediana no era la asimetria: era el filtro IQR.")


y_train (FILTRADO): media=4.0869  mediana=3.0  skew=0.985  max=12
y_test  (sin filtrar): media=4.3959  mediana=4.0  skew=1.134  max=14



y_train SIN filtrar: media=4.3960  mediana=4.0  max=14

MAE(test) prediciendo la mediana del train FILTRADO (3.0):      2.2929
MAE(test) prediciendo la media del train filtrado (4.087):    2.2805
MAE(test) prediciendo la mediana del train SIN filtrar (4.0):   2.2596   <- PISO HONESTO
MAE(test) prediciendo la mediana del TEST (4.0):              2.2596   <- optimo entre constantes

La mediana del train sin filtrar (4.0) coincide con la del test y alcanza el
optimo entre constantes SIN mirar el test: el optimo era accesible todo el tiempo.
Lo que degradaba al baseline de mediana no era la asimetria: era el filtro IQR.


In [5]:
# Se registra el piso honesto como cuarto baseline: la mejor constante que se
# puede elegir con informacion de entrenamiento legitima.

class ConstanteMedianaSinFiltrar:
    """Predictor constante = mediana del train ANTES del filtro IQR."""

    def __init__(self, valor):
        self.valor = float(valor)

    def predict(self, X):
        return np.full(len(X), self.valor)

piso = ConstanteMedianaSinFiltrar(mediana_sin_filtrar)

# mae_cv con el mismo protocolo: la constante es fija, asi que su MAE de CV es
# el promedio sobre los folds del train filtrado.
maes_fold = []
for _, idx_val in cv.split(X_train):
    maes_fold.append((y_train.iloc[idx_val] - piso.valor).abs().mean())

metricas_piso = evaluar(piso, X_test, y_test)
metricas_piso["mae_cv"] = float(np.mean(maes_fold))
metricas_piso["tiempo_s"] = 0.0
registrar(
    "baseline_mediana_sin_filtrar", metricas_piso,
    notas=(f"Constante = mediana del train SIN filtro IQR ({piso.valor:.1f} dias); "
           "piso honesto: iguala el optimo entre constantes sin mirar el test"),
)
print({k: round(v, 4) for k, v in metricas_piso.items()})
print()
print("Este es el piso que los modelos de los NB03-06 deben superar de verdad.")


{'mae': 2.2596, 'rmse': 3.0111, 'r2': -0.0176, 'pred_test_s': 0.0, 'mae_cv': 2.0429, 'tiempo_s': 0.0}

Este es el piso que los modelos de los NB03-06 deben superar de verdad.


## Familias consideradas y descartadas (criterio oficial 5)

El criterio 5 pide justificar la elección *por sobre otras opciones*. Las cinco familias que siguen (KNN, SVR, árbol, Random Forest, XGBoost) cubren el mapa paramétrico↔no paramétrico y local↔global, pero hay tres alternativas naturales para este problema que **se evaluaron conceptualmente y se descartaron**. Dejarlas sin mencionar sería presentar el temario de la materia como si fuera el resultado de una deliberación:

| Familia | Por qué sería candidata | Por qué se descarta |
|---|---|---|
| **GLM de Poisson / binomial negativa** | El target es un **conteo discreto** (1–14 días) y asimétrico a derecha: exactamente el caso para el que se diseñaron estos modelos, que además garantizan predicciones positivas. | Es la alternativa más defendible y su ausencia es una limitación real, no una victoria del diseño. Se descarta por alcance: la materia es de aprendizaje automático y no cubre GLM, y el trabajo prioriza comparar familias vistas en clase bajo un protocolo idéntico. **Queda registrada como camino futuro en el NB07.** Nota: XGBoost con `objective="count:poisson"` daría una aproximación barata dentro del stack ya montado. |
| **Regresión cuantílica** | Optimiza directamente una pérdida de tipo valor absoluto (la mediana condicional minimiza el MAE, nuestra métrica primaria) y daría intervalos de predicción, útiles para planificar capacidad con un percentil de seguridad. | El objetivo declarado es un **estimador puntual** de días de estadía comparable entre familias. El beneficio (intervalos) se logra igual con `reg:absoluteerror` en XGBoost —que sí se usa, decisión D4 del NB06— más un bootstrap posterior. |
| **Regresión ordinal** | El target podría leerse como 14 categorías ordenadas en lugar de un número real. | Perdería la métrica de negocio: la gestión de camas necesita *días*, y el MAE sobre días es directamente interpretable. Además desperdicia la información de que la distancia entre 2 y 3 días es la misma que entre 12 y 13. |

Las cinco familias efectivamente comparadas se justifican una por una en la celda de apertura de los notebooks 03 a 06, con el argumento anclado en **estos** datos (dimensionalidad, tipos de variables, tamaño de muestra y costo de predicción) y no en propiedades genéricas de manual.

## Lectura de negocio

- **El piso honesto a superar es MAE = 2,2596 días**, no 2,2805. La política ingenua "asumir que todo paciente se queda lo que el paciente típico" se equivoca, en el mejor de los casos, 2,26 días por paciente. Todo modelo de los notebooks 03–06 se juzga por cuántos días le recorta a ese piso — y usarlo, en lugar del `DummyRegressor(mean)` que quedó 0,02 días peor por un artefacto del filtro, hace la comparación más exigente y no menos.
- **Qué mostró la anomalía media-vs-mediana.** La media (2,2805) le ganó a la mediana (2,2929) en la métrica que la mediana debería minimizar. La causa no es la asimetría del target sino el filtro IQR del notebook 01: al recortar el train, corrió su mediana de 4 a 3 días. Con el train sin filtrar la mediana vuelve a 4,0 y alcanza el óptimo entre constantes. Es la primera evidencia cuantificada de que **el filtro heredado tiene un costo medible**, y anticipa la ablación completa del notebook 07.
- La **regresión lineal** baja el MAE a 1,86 días con R² = 0,32: hay señal lineal real en las 17 features, pero dos tercios de la varianza siguen sin explicarse — ese es el espacio que los modelos no lineales (KNN, SVR, árboles, ensambles) intentarán capturar.
- Contexto operativo: cada décima de día que los modelos siguientes recorten, a escala de un hospital con miles de admisiones anuales, es error de planificación de camas que desaparece.

**Siguiente notebook (03):** KNN regressor — primer modelo real, con la justificación de por qué exige escalado.